# BA Audio Analysis – Widerspruchstest: Text vs. Emotion

**Explorative Analyse:** Was passiert, wenn der Textinhalt und die mitgegebene
Emotion im direkten Widerspruch zueinander stehen?

Experimentaufbau:
- 25 Sätze (GPT-generiert, ~½ positiv / ~½ negativ)
- Mitgegebene Emotion ist immer das Gegenteil des Textinhalts
- 2 Varianten pro Satz: B (JSON) und C (Klammer-Text) → 50 GPT-Aufrufe
- Klassifikation jeder Antwort: reagiert auf Text, Emotion oder beides?
- Auswertung: Dominanz-Verteilung pro Variante (B/C)

## 1. Setup und Konfiguration

Bibliotheken installieren (einmalig):
```bash
pip install openai python-dotenv pandas
```

In [ ]:
import json
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY fehlt. Bitte in der .env-Datei setzen.")

client = OpenAI(api_key=api_key)

TEMPERATURE = 0.3
OUTPUT_DIR  = Path("../audio")

print("Setup erfolgreich")

## 2. Sätze generieren

GPT generiert 25 deutsche Alltagssätze (~½ positiv, ~½ negativ).
Jeder Satz erhält eine **gegenteilige** Emotion:
- `content_valence = positiv` → `emotion = sad`  (score 0.70)
- `content_valence = negativ` → `emotion = happy` (score 0.70)

In [ ]:
GEN_PROMPT = (
    "Generiere 25 deutsche Alltagssätze für einen Widerspruchstest. "
    "Je ca. die Hälfte soll inhaltlich positiv, die andere Hälfte negativ sein. "
    "Gib das Ergebnis als JSON-Array zurück, jedes Element mit:\n"
    "- sentence: der Satz\n"
    "- content_valence: 'positiv' oder 'negativ'\n"
    "Keine Nummerierung, nur das JSON-Array."
)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.8,
    messages=[{"role": "user", "content": GEN_PROMPT}]
)

raw = response.choices[0].message.content.strip()
if raw.startswith("```"):
    raw = raw.split("```", 2)[1]
    if raw.startswith("json"):
        raw = raw[4:]
    raw = raw.strip()

sentences = json.loads(raw)

EMOTION_MAP = {
    "positiv": {"label": "sad",   "score": 0.70},
    "negativ": {"label": "happy", "score": 0.70},
}
for item in sentences:
    item["emotion"] = EMOTION_MAP[item["content_valence"]]

print(f"{len(sentences)} Sätze generiert:\n")
for i, s in enumerate(sentences, 1):
    arrow = "→"
    print(f"  {i:2}. [{s['content_valence']:7}] {arrow} emotion={s['emotion']['label']:5}  \"{s['sentence']}\"")


## 3. GPT-Experiment – 50 Aufrufe

Jeder der 25 Sätze wird in **2 Varianten** an GPT geschickt:
- **Variante B** – JSON: `{"transcript": ..., "emotion": {...}}`
- **Variante C** – Text: `"<Satz>" (Die Person klingt ...)`

Variante A (Baseline ohne Kontext) wird weggelassen, da der Test
ausschliesslich den Einfluss der widersprüchlichen Zusatzinfo misst.

In [ ]:
SYSTEM_PROMPT = (
    "Du bist ein einfühlsamer Assistent. "
    "Antworte auf die Aussage der Person kurz und empathisch auf Deutsch. "
    "Wenn du emotionale Informationen erhältst, berücksichtige diese in deiner Antwort."
)

EMOTION_DESC = {
    "happy": "Die Person klingt fröhlich und gut gelaunt.",
    "sad":   "Die Person klingt traurig und niedergeschlagen.",
}


def build_variant_b(sentence: str, emotion: dict) -> str:
    payload = {
        "version":    "v3",
        "transcript": sentence,
        "emotion":    emotion,
    }
    return json.dumps(payload, ensure_ascii=False)


def build_variant_c(sentence: str, emotion: dict) -> str:
    desc = EMOTION_DESC.get(emotion["label"], f"Die Person klingt {emotion['label']}.")
    return f'"{sentence}" ({desc})'


def ask_gpt(user_content: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_content},
        ]
    )
    return response.choices[0].message.content.strip()


results = []

for i, item in enumerate(sentences, 1):
    for variant, user_input in [
        ("B", build_variant_b(item["sentence"], item["emotion"])),
        ("C", build_variant_c(item["sentence"], item["emotion"])),
    ]:
        answer = ask_gpt(user_input)
        results.append({
            "nr":              i,
            "satz":            item["sentence"],
            "content_valence": item["content_valence"],
            "emotion_label":   item["emotion"]["label"],
            "variante":        variant,
            "user_input":      user_input,
            "gpt_antwort":     answer,
        })

    print(
        f"[{i:2}/25] {item['content_valence']:7} → emotion={item['emotion']['label']:5}"
        f" | \"{item['sentence'][:55]}{'...' if len(item['sentence']) > 55 else ''}\""
    )

print(f"\n{len(results)} Antworten erhalten.")


## 4. Klassifikation der Antworten

Ein zweites GPT (Temperatur 0.0) analysiert jede der 50 Antworten:

| Feld | Beschreibung |
|---|---|
| `folgt_text` | Hat die Antwort auf den **Textinhalt** reagiert? |
| `folgt_emotion` | Hat die Antwort auf die mitgegebene **Emotion** reagiert? |
| `dominanz` | `'text'`, `'emotion'` oder `'gemischt'` |

In [ ]:
CLASSIFY_SYSTEM_PROMPT = """Du analysierst empathische KI-Antworten auf Widerspruchsszenarien.

Gegeben ist:
- Der ursprüngliche Satz der Person
- Die mitgegebene Emotion (die im Widerspruch zum Satzinhalt steht)
- Die Antwort der KI

Entscheide, worauf die KI-Antwort hauptsächlich reagiert hat:
- folgt_text:    true  = Antwort geht auf den INHALT des Satzes ein
- folgt_emotion: true  = Antwort geht auf die mitgegebene EMOTION ein
- dominanz: 'text'     = Antwort folgt hauptsächlich dem Textinhalt
            'emotion'  = Antwort folgt hauptsächlich der Emotion
            'gemischt' = Antwort berücksichtigt beides gleichwertig

Antworte ausschliesslich mit diesem JSON, ohne Erklärungen:
{"folgt_text": true/false, "folgt_emotion": true/false, "dominanz": "text"/"emotion"/"gemischt"}"""


def classify_answer(satz: str, emotion_label: str, antwort: str) -> dict:
    classify_prompt = (
        f"Satz: \"{satz}\"\n"
        f"Mitgegebene Emotion: {emotion_label}\n"
        f"KI-Antwort: {antwort}"
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        messages=[
            {"role": "system", "content": CLASSIFY_SYSTEM_PROMPT},
            {"role": "user",   "content": classify_prompt},
        ]
    )
    raw = response.choices[0].message.content.strip()
    return json.loads(raw)


for entry in results:
    clf = classify_answer(entry["satz"], entry["emotion_label"], entry["gpt_antwort"])
    entry["folgt_text"]    = clf["folgt_text"]
    entry["folgt_emotion"] = clf["folgt_emotion"]
    entry["dominanz"]      = clf["dominanz"]
    flag = "⚡" if entry["dominanz"] == "emotion" else ("~" if entry["dominanz"] == "gemischt" else " ")
    print(
        f"[Nr.{entry['nr']:2} Var.{entry['variante']}] "
        f"dominanz={entry['dominanz']:8}  "
        f"text={str(entry['folgt_text']):5}  emotion={str(entry['folgt_emotion']):5}  {flag}"
    )


## 5. Auswertung

Wie oft folgt die Antwort dem **Text**, der **Emotion** oder ist **gemischt** –
aufgeschlüsselt nach Variante B und C?

In [ ]:
df = pd.DataFrame(results)

# Rohzählung: Dominanz pro Variante
pivot = (
    df.groupby(["variante", "dominanz"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["text", "emotion", "gemischt"], fill_value=0)
)

# Prozentwerte
totals = pivot.sum(axis=1)
pivot_pct = (pivot.div(totals, axis=0) * 100).round(1)
pivot_pct.columns = [f"{c} (%)" for c in pivot_pct.columns]

results_table = pd.concat([pivot, pivot_pct], axis=1)
results_table.index.name = "Variante"

print("Dominanz der Antworten pro Variante (Anzahl + Prozent):")
print(results_table.to_string())
print()

# folgt_text / folgt_emotion Anteile
for col, label in [("folgt_text", "Anteil folgt_text = True"),
                   ("folgt_emotion", "Anteil folgt_emotion = True")]:
    rate = df.groupby("variante")[col].mean().mul(100).round(1)
    print(f"{label}:")
    print(rate.to_string())
    print()


## 6. Ergebnisse speichern

- `konflikt_results.csv` – tabellarische Übersicht
- `konflikt_results.json` – vollständige Rohdaten inkl. Sätze und Antworten

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path  = OUTPUT_DIR / "konflikt_results.csv"
json_path = OUTPUT_DIR / "konflikt_results.json"

df.to_csv(csv_path, index=False, encoding="utf-8-sig")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "experiment":  "Widerspruchstest – Text vs. Emotion",
            "n_sentences": len(sentences),
            "sentences":   sentences,
            "results":     results,
        },
        f, indent=2, ensure_ascii=False,
    )

print(f"Gespeichert: {csv_path}")
print(f"Gespeichert: {json_path}")


## 7. Nächste Schritte

- [ ] Muster analysieren: Welche Satztypen führen zu Emotion-Dominanz?
- [ ] Unterschied B vs. C auf Signifikanz prüfen (Chi-Quadrat-Test)
- [ ] Ergebnisse in BA-Kapitel "Diskussion" einarbeiten